In [3]:
import numpy as np
from scipy.sparse import eye, lil_matrix
from resource_estimate_utils import *
from os.path import join
from time import time
import networkx as nx

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.synthesis import LieTrotter, SuzukiTrotter
from qiskit import transpile
from qiskit.circuit.library import PauliEvolutionGate
from pytket import OpType
from pytket.passes import RemoveRedundancies, CommuteThroughMultis, SequencePass, FullPeepholeOptimise, auto_rebase_pass
from pytket.extensions.qiskit import qiskit_to_tk
from braket.devices import LocalSimulator

from os.path import join, dirname
from utils import *

In [6]:
def lchs_off_diagonal_circuit(n, k1, k2, t, J, h, gamma, r=1, compute_real_part=True):
    assert k1 != k2

    circuit = QuantumCircuit(n+1)
    circuit.h(0)
    if not compute_real_part:
        circuit.rz(-np.pi/2, 0)

    dt = t / r
    # Second-order Trotter
    for _ in range(r):
        # TFIM part
        for i in range(n):
            circuit.rx(dt * h, 1 + i)
        for i in range(n-1):
            circuit.rzz(dt * J, 1 + i, 1 + i + 1)

        # Anti-Hermitian part
        a = -2 * dt * gamma * (k1 + k2) / 2
        b = -2 * dt * gamma * (k1 - k2) / 2
        for i in range(n):

            circuit.rz(-b, 0)
            circuit.rz(a, 1 + i)
            circuit.h(0)
            circuit.h(1 + i)
            circuit.rxx(b, 0, 1 + i)
            circuit.h(0)
            circuit.h(1 + i)

        # TFIM part
        for i in range(n):
            circuit.rx(dt * h, 1 + i)
        for i in range(n-1):
            circuit.rzz(dt * J, 1 + i, 1 + i + 1)
    
    circuit.h(0)
    return circuit

def get_lchs_hamiltonian(n, J, h, gamma, k1, k2):

    pauli_op_list = []
    # Hermitian part
    for i in range(n):
        op = (n+1) * ['I']
        op[i] = 'X'
        pauli_op_list.append((''.join(op), h))
    for i in range(n-1):
        op = (n+1) * ['I']
        op[i] = 'Z'
        op[i+1] = 'Z'
        pauli_op_list.append((''.join(op), h))

    # Anti-Hermitian part
    for i in range(n):
        op = (n+1) * ['I']
        op[i] = 'Z'
        op[n] = 'Z'
        pauli_op_list.append((''.join(op), k1 * gamma))
        op[i] = 'Z'
        op[n] = 'Z'
        pauli_op_list.append((''.join(op), k1 * gamma))

    print(pauli_op_list)

In [5]:
T = 2
J = 1
h = 1
gamma = 0.05
R = 128
n_p = 8
N_p = 2 ** n_p
print(f"N_p={N_p}")

error_tol = 5e-2
trotter_method = "second_order"



n_vals = np.arange(2, 10)


# Resource analysis for LCHS
for i, n in enumerate(n_vals):
    k1 = R
    k2 = -R
    circuit = lchs_off_diagonal_circuit(n, k1, k2, T, J, h, gamma, r=1, compute_real_part=True):


# Resource analysis for Schrodingerization

256
